# Text-to-Speech Model Development and Inference Experiments

This notebook documents environment setup, inspection and loading of pretrained FastSpeech 2 and HiFi-GAN checkpoints, speech inference tests, and experimental FastAPI/Cloudflare tunnel integration. It does **not** contain a model-training workflow; the notebook uses existing pretrained checkpoints.

**Security note:** Saved cell outputs have been cleared to remove generated API-key values and temporary tunnel URLs. API keys should be generated at runtime and never printed or committed.

**Note:** The notebook contains iterative development cells. Run only the relevant cells in order for the experiment you want to reproduce; Colab paths and temporary tunnel URLs may need updating.


In [ ]:

import torch

print("PyTorch Version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU Available: Yes")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU Available: No")
    print("Please check your runtime settings.")


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


In [ ]:

import os

drive_path = "/content/drive/MyDrive"

print("Files and folders in Google Drive:\n")

for item in os.listdir(drive_path):
    print(item)


In [ ]:
import os

project_path = "/content/drive/MyDrive/Text_to_Speech_Project"

print("Contents of Text_to_Speech_Project:\n")

for item in os.listdir(project_path):
    print(item)

In [ ]:
import os

drive_path = "/content/drive/MyDrive"

keywords = ["fastspeech", "hifi", "checkpoint"]

print("Searching Google Drive for model files...\n")

for root, dirs, files in os.walk(drive_path):
    for item in dirs + files:
        if any(word in item.lower() for word in keywords):
            print(os.path.join(root, item))

In [ ]:
import os

project_path = "/content/drive/MyDrive/Text_to_Speech_Project"

for root, dirs, files in os.walk(project_path):
    level = root.replace(project_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  {file}")

In [ ]:
import os

drive_path = "/content/drive/MyDrive"

print("Google Drive folders and files:\n")

for item in os.listdir(drive_path):
    full_path = os.path.join(drive_path, item)

    if os.path.isdir(full_path):
        print("[FOLDER]", item)
    else:
        print("[FILE]", item)

In [ ]:
import os

project_path = "/content/drive/MyDrive/Text_to_Speech_Project"

folders = ["FastSpeech 2", "HiFi-GAN"]

for folder in folders:
    folder_path = os.path.join(project_path, folder)

    print(f"\nChecking: {folder}")

    if os.path.exists(folder_path):
        print("Status: Found successfully")
        print("Contents:")

        for item in os.listdir(folder_path)[:20]:
            print(" -", item)
    else:
        print("Status: Folder not found")

In [ ]:
import os

project_path = "/content/drive/MyDrive/Text_to_Speech_Project"

print("Folders and files inside Text_to_Speech_Project:\n")

if os.path.exists(project_path):
    for item in os.listdir(project_path):
        full_path = os.path.join(project_path, item)

        if os.path.isdir(full_path):
            print("[FOLDER]", repr(item))
        else:
            print("[FILE]", repr(item))
else:
    print("Project folder not found. Check Google Drive mounting.")

In [ ]:
import os

project_path = "/content/drive/MyDrive/Text_to_Speech_Project"

for folder in ["fastspeech2", "hifigan"]:
    folder_path = os.path.join(project_path, folder)

    print(f"\n{'='*50}")
    print(f"FOLDER: {folder}")
    print('='*50)

    for root, dirs, files in os.walk(folder_path):
        level = root.replace(folder_path, "").count(os.sep)

        if level > 2:
            dirs[:] = []
            continue

        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")

        for file in files[:15]:
            print(f"{indent}  {file}")

In [ ]:
import os

project_path = "/content/drive/MyDrive/Text_to_Speech_Project"

extensions = (
    ".pt", ".pth", ".ckpt", ".bin",
    ".yaml", ".yml", ".json",
    ".safetensors"
)

for folder in ["fastspeech2", "hifigan"]:
    folder_path = os.path.join(project_path, folder)

    print(f"\n{'='*45}")
    print(f"SEARCHING: {folder}")
    print('='*45)

    found = False

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(extensions):
                print(os.path.relpath(os.path.join(root, file), folder_path))
                found = True

    if not found:
        print("No checkpoint or configuration files found.")

In [ ]:
import torch
import os

print("========== PROJECT ENVIRONMENT ==========")

print("PyTorch Version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU Available: Yes")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU Available: No")

project_path = "/content/drive/MyDrive/Text_to_Speech_Project"

print("\n========== PROJECT FOLDER ==========")

if os.path.exists(project_path):
    print("Project folder found!")

    for item in os.listdir(project_path):
        print("-", item)
else:
    print("Project folder not found. Drive may need mounting.")

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is still not available.")

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Text_to_Speech_Project/LJSpeech-1.1"

print("========== LJSPEECH DATASET CHECK ==========")

if os.path.exists(dataset_path):
    print("Dataset found!")

    print("\nFiles and folders:")

    for item in os.listdir(dataset_path):
        full_path = os.path.join(dataset_path, item)

        if os.path.isdir(full_path):
            print("[FOLDER]", item)
        else:
            print("[FILE]", item)

else:
    print("Dataset folder not found!")

In [ ]:
import os

project_path = "/content/drive/MyDrive/Text_to_Speech_Project"

print("========== PROJECT CONTENTS ==========")

for item in os.listdir(project_path):
    full_path = os.path.join(project_path, item)

    print(
        repr(item),
        "| Folder:", os.path.isdir(full_path),
        "| Path:", full_path
    )

print("\n========== SEARCHING FOR LJSPEECH ==========")

for root, dirs, files in os.walk(project_path):
    for folder in dirs:
        if "ljspeech" in folder.lower():
            print("Dataset folder found:", os.path.join(root, folder))

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Text_to_Speech_Project/LJSpeech-1.1"

print("========== DATASET CONTENTS ==========")

for item in os.listdir(dataset_path):
    full_path = os.path.join(dataset_path, item)

    if os.path.isdir(full_path):
        print("[FOLDER]", item)

        if item.lower() == "wavs":
            wav_files = [
                f for f in os.listdir(full_path)
                if f.lower().endswith(".wav")
            ]
            print("Total WAV files:", len(wav_files))
            print("Sample files:", wav_files[:5])

    else:
        print("[FILE]", item)

In [ ]:
import pandas as pd
import os

dataset_path = "/content/drive/MyDrive/Text_to_Speech_Project/LJSpeech-1.1"

metadata_path = os.path.join(dataset_path, "metadata.csv")

metadata = pd.read_csv(
    metadata_path,
    sep="|",
    header=None,
    names=["file_id", "transcription", "normalized_text"]
)

print("========== METADATA VALIDATION ==========")

print("Total metadata records:", len(metadata))

print("\nFirst 5 records:")
print(metadata.head())

print("\nMissing values:")
print(metadata.isnull().sum())

print("\nDuplicate audio IDs:", metadata["file_id"].duplicated().sum())

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Text_to_Speech_Project/LJSpeech-1.1"

wavs_path = os.path.join(dataset_path, "wavs")

missing_audio = []

for file_id in metadata["file_id"]:
    audio_path = os.path.join(wavs_path, file_id + ".wav")

    if not os.path.exists(audio_path):
        missing_audio.append(file_id)

print("========== AUDIO-TEXT MATCHING ==========")

print("Total transcript records:", len(metadata))
print("Total audio files:", len(os.listdir(wavs_path)))
print("Missing audio files:", len(missing_audio))

if missing_audio:
    print("\nFirst missing IDs:", missing_audio[:10])
else:
    print("\nAll metadata records have matching audio files!")

In [ ]:
import os
import soundfile as sf
from IPython.display import Audio, display

dataset_path = "/content/drive/MyDrive/Text_to_Speech_Project/LJSpeech-1.1"

audio_path = os.path.join(
    dataset_path,
    "wavs",
    metadata.iloc[0]["file_id"] + ".wav"
)

audio, sample_rate = sf.read(audio_path)

print("========== AUDIO PROPERTIES ==========")

print("Audio file:", os.path.basename(audio_path))
print("Sampling rate:", sample_rate, "Hz")
print("Duration:", round(len(audio) / sample_rate, 2), "seconds")
print("Audio shape:", audio.shape)
print("Number of channels:", 1 if audio.ndim == 1 else audio.shape[1])
print("Audio data type:", audio.dtype)

print("\nTranscript:")
print(metadata.iloc[0]["transcription"])

print("\nPlaying sample audio:")
display(Audio(audio, rate=sample_rate))

In [ ]:
import numpy as np
import soundfile as sf
import os

dataset_path = "/content/drive/MyDrive/Text_to_Speech_Project/LJSpeech-1.1"
wavs_path = os.path.join(dataset_path, "wavs")

audio_files = [
    f for f in os.listdir(wavs_path)
    if f.lower().endswith(".wav")
]

durations = []
invalid_files = []

for filename in audio_files:
    path = os.path.join(wavs_path, filename)

    try:
        info = sf.info(path)

        if info.samplerate != 22050 or info.channels != 1:
            invalid_files.append(filename)

        durations.append(info.duration)

    except Exception:
        invalid_files.append(filename)

print("========== DATASET AUDIO STATISTICS ==========")

print("Total WAV files:", len(audio_files))
print("Average duration:", round(np.mean(durations), 2), "seconds")
print("Shortest duration:", round(np.min(durations), 2), "seconds")
print("Longest duration:", round(np.max(durations), 2), "seconds")
print("Total audio duration:", round(sum(durations) / 3600, 2), "hours")
print("Invalid audio files:", len(invalid_files))

In [ ]:
import os

model_path = "/content/drive/MyDrive/Text_to_Speech_Project/fastspeech2"

print("========== FASTSPEECH 2 SOURCE FILES ==========")

for filename in os.listdir(model_path):
    filepath = os.path.join(model_path, filename)

    if filename.endswith(".py"):
        print(f"\n{'='*50}")
        print("FILE:", filename)
        print('='*50)

        with open(filepath, "r", encoding="utf-8") as file:
            content = file.readlines()

        print("Total lines:", len(content))

        for line_number, line in enumerate(content[:15], start=1):
            print(f"{line_number}: {line.rstrip()}")

In [ ]:
import os

check_paths = [
    "/content/drive/MyDrive/Text_to_Speech_Project",
    "/content/drive/MyDrive/Text_to_Speech_Project/fastspeech2",
    "/content/drive/MyDrive/Text_to_Speech_Project/hifigan",
]

for path in check_paths:
    print("\nFolder:", path)
    print("-" * 60)

    if os.path.exists(path):
        for item in os.listdir(path):
            print(item)
    else:
        print("Folder not found")

In [ ]:
import importlib.util

modules_to_check = [
    "models",
    "modules",
    "utils",
    "optimizer"
]

for module in modules_to_check:
    spec = importlib.util.find_spec(module)
    print(f"{module}:",
          "Available" if spec else "Not available")

In [ ]:
%cd /content

!git clone https://github.com/open-mmlab/Amphion.git

In [ ]:
import os

amphion_path = "/content/Amphion"

folders_to_check = [
    "models",
    "modules",
    "utils",
    "optimizer",
    "egs/tts/FastSpeech2"
]

for folder in folders_to_check:
    path = os.path.join(amphion_path, folder)

    print(
        folder,
        "->",
        "Available" if os.path.exists(path) else "Missing"
    )

In [ ]:
import os

example_path = "/content/Amphion/egs/tts/FastSpeech2"

for root, dirs, files in os.walk(example_path):
    level = root.replace(example_path, "").count(os.sep)

    if level > 2:
        dirs[:] = []
        continue

    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for filename in files:
        print(f"{indent}    {filename}")


In [ ]:
config_path = "/content/Amphion/egs/tts/FastSpeech2/exp_config.json"

with open(config_path, "r", encoding="utf-8") as file:
    print(file.read())

In [ ]:
readme_path = "/content/Amphion/egs/tts/FastSpeech2/README.md"

with open(readme_path, "r", encoding="utf-8") as file:
    print(file.read())

In [ ]:
script_path = "/content/Amphion/egs/tts/FastSpeech2/run.sh"

with open(script_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total lines:", len(lines))
print("\n".join(
    f"{i+1}: {line.rstrip()}"
    for i, line in enumerate(lines)
))

In [ ]:
import shutil
import subprocess

print("========== DOWNLOAD READINESS ==========")

result = subprocess.run(
    ["git", "lfs", "version"],
    capture_output=True,
    text=True
)

print("Git LFS:", result.stdout.strip()
      if result.returncode == 0 else "Not installed")

total, used, free = shutil.disk_usage("/content")

print(f"Total storage: {total / (1024**3):.2f} GB")
print(f"Used storage:  {used / (1024**3):.2f} GB")
print(f"Free storage:  {free / (1024**3):.2f} GB")

In [ ]:
import importlib.util

if importlib.util.find_spec("huggingface_hub"):
    import huggingface_hub
    print("Hugging Face Hub is installed.")
    print("Version:", huggingface_hub.__version__)
else:
    print("Hugging Face Hub is not installed.")

In [ ]:
from huggingface_hub import snapshot_download

fastspeech_path = snapshot_download(
    repo_id="amphion/fastspeech2_ljspeech",
    local_dir="/content/fastspeech2_ljspeech"
)

print("\nFastSpeech 2 download completed!")
print("Location:", fastspeech_path)

In [ ]:
import os

model_dir = "/content/fastspeech2_ljspeech"

print("========== FASTSPEECH 2 MODEL FILES ==========")

for root, dirs, files in os.walk(model_dir):
    level = root.replace(model_dir, "").count(os.sep)
    indent = "    " * level

    print(f"\n{indent}{os.path.basename(root)}/")

    for filename in files:
        filepath = os.path.join(root, filename)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)

        print(f"{indent}    {filename} ({size_mb:.2f} MB)")

In [ ]:
from huggingface_hub import snapshot_download

hifigan_path = snapshot_download(
    repo_id="amphion/hifigan_ljspeech",
    local_dir="/content/hifigan_ljspeech"
)

print("\nHiFi-GAN download completed!")
print("Location:", hifigan_path)

In [ ]:
import os

model_dir = "/content/hifigan_ljspeech"

print("========== HIFIGAN MODEL FILES ==========")

for root, dirs, files in os.walk(model_dir):
    dirs[:] = [d for d in dirs if d != ".cache"]

    level = root.replace(model_dir, "").count(os.sep)
    indent = "    " * level

    print(f"\n{indent}{os.path.basename(root)}/")

    for filename in files:
        filepath = os.path.join(root, filename)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)

        print(f"{indent}    {filename} ({size_mb:.2f} MB)")

In [ ]:
config_files = {
    "FastSpeech 2": "/content/fastspeech2_ljspeech/args.json",
    "HiFi-GAN": "/content/hifigan_ljspeech/args.json"
}

for model_name, config_path in config_files.items():

    print("\n" + "=" * 60)
    print(model_name.upper())
    print("=" * 60)

    with open(config_path, "r", encoding="utf-8") as file:
        print(file.read())

In [ ]:
from pathlib import Path

config_file = Path("/content/Amphion/utils/util.py")

print("Checking Amphion configuration loader...")
print("File exists:", config_file.exists())

if config_file.exists():
    lines = config_file.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines):
        if "load_config" in line or "json.load" in line or "json5" in line:
            start = max(0, i - 5)
            end = min(len(lines), i + 15)

            print("\n" + "=" * 50)
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")

In [ ]:
%pip install json5

In [ ]:
%pip install ruamel.yaml

In [ ]:
import os

search_root = "/content/Amphion"

print("Searching for fs2.json...\n")

matches = []

for root, dirs, files in os.walk(search_root):
    if "fs2.json" in files:
        matches.append(os.path.join(root, "fs2.json"))

if matches:
    for path in matches:
        print("FOUND:", path)
else:
    print("fs2.json was not found in the Amphion repository.")

print("\nFastSpeech 2 args.json base configuration:")

import json5

args_path = "/content/fastspeech2_ljspeech/args.json"

with open(args_path, "r", encoding="utf-8") as f:
    args = json5.load(f)

print(args.get("base_config", "No base_config entry found"))

In [ ]:
import os
import sys

os.chdir("/content/Amphion")
sys.path.insert(0, "/content/Amphion")

from utils.util import load_config

config_paths = {
    "FastSpeech 2": "/content/fastspeech2_ljspeech/args.json",
    "HiFi-GAN": "/content/hifigan_ljspeech/args.json"
}

for model_name, config_path in config_paths.items():

    print("\n" + "=" * 60)
    print(model_name.upper())
    print("=" * 60)

    config = load_config(config_path)

    print("Configuration loaded successfully!")
    print("Configuration keys:")
    print(list(config.keys()))

print("\nBoth configuration files checked.")

In [ ]:
from utils.util import load_config

fs2 = load_config(
    "/content/fastspeech2_ljspeech/args.json"
)

hifigan = load_config(
    "/content/hifigan_ljspeech/args.json"
)

def show_config(config, name):
    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    for section in ["preprocess", "model"]:
        if hasattr(config, section):
            values = getattr(config, section)

            print(f"\n{section.upper()} SETTINGS")

            if hasattr(values, "keys"):
                for key in values.keys():
                    value = values[key]

                    if any(word in key.lower() for word in [
                        "sample_rate", "sampling_rate",
                        "mel", "fft", "hop", "win",
                        "fmin", "fmax"
                    ]):
                        print(f"{key}: {value}")

show_config(fs2, "FASTSPEECH 2")
show_config(hifigan, "HIFI-GAN")

In [ ]:
import os

checkpoints = {
    "FastSpeech 2": (
        "/content/fastspeech2_ljspeech/checkpoint/"
        "epoch-0125_step-0097902_loss-1.074404/"
        "pytorch_model.bin"
    ),

    "HiFi-GAN": (
        "/content/hifigan_ljspeech/checkpoints/"
        "step-0560000_loss-51.2373.pt"
    )
}

print("=" * 60)
print("PRETRAINED CHECKPOINT VERIFICATION")
print("=" * 60)

for name, path in checkpoints.items():

    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", os.path.isfile(path))

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"File size: {size_mb:.2f} MB")
    else:
        print("ERROR: Checkpoint file is missing!")

In [ ]:
from pathlib import Path

files_to_check = [
    Path("/content/Amphion/bins/tts/inference.py"),
    Path("/content/Amphion/egs/tts/FastSpeech2/run.sh")
]

keywords = [
    "torch.load",
    "load_state_dict",
    "checkpoint",
    "vocoder",
    "FastSpeech2",
    "HiFiGAN",
    "load_model"
]

for file_path in files_to_check:

    print("\n" + "=" * 65)
    print("FILE:", file_path.name)
    print("=" * 65)

    lines = file_path.read_text(encoding="utf-8").splitlines()

    matching_lines = [
        i for i, line in enumerate(lines)
        if any(word.lower() in line.lower() for word in keywords)
    ]

    shown = set()

    for i in matching_lines:
        start = max(0, i - 3)
        end = min(len(lines), i + 5)

        for j in range(start, end):
            if j not in shown:
                print(f"{j + 1}: {lines[j]}")
                shown.add(j)

        print("-" * 35)

In [ ]:
from pathlib import Path

files_to_check = [
    Path("/content/Amphion/models/tts/fastspeech2/fs2_inference.py"),
    Path("/content/Amphion/models/vocoders/hifigan/hifigan_inference.py")
]

for file_path in files_to_check:

    print("\n" + "=" * 65)
    print("FILE:", file_path)
    print("=" * 65)

    if not file_path.exists():
        print("File not found.")
        continue

    lines = file_path.read_text(encoding="utf-8").splitlines()

    keywords = [
        "torch.load",
        "load_state_dict",
        "checkpoint",
        "vocoder",
        "def __init__",
        "def load",
        "def inference"
    ]

    shown = set()

    for i, line in enumerate(lines):
        if any(word.lower() in line.lower() for word in keywords):

            start = max(0, i - 3)
            end = min(len(lines), i + 7)

            for j in range(start, end):
                if j not in shown:
                    print(f"{j + 1}: {lines[j]}")
                    shown.add(j)

            print("-" * 40)

In [ ]:
import torch

fs2_path = (
    "/content/fastspeech2_ljspeech/checkpoint/"
    "epoch-0125_step-0097902_loss-1.074404/"
    "pytorch_model.bin"
)

hifigan_path = (
    "/content/hifigan_ljspeech/checkpoints/"
    "step-0560000_loss-51.2373.pt"
)

print("=" * 60)
print("FASTSPEECH 2 CHECKPOINT")
print("=" * 60)

fs2_checkpoint = torch.load(
    fs2_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(fs2_checkpoint))

if isinstance(fs2_checkpoint, dict):
    print("Top-level keys:", list(fs2_checkpoint.keys()))

print("\n" + "=" * 60)
print("HIFI-GAN CHECKPOINT")
print("=" * 60)

hifi_checkpoint = torch.load(
    hifigan_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(hifi_checkpoint))

if isinstance(hifi_checkpoint, dict):
    print("Top-level keys:", list(hifi_checkpoint.keys()))

In [ ]:
from pathlib import Path

search_files = [
    Path("/content/Amphion/models/tts/tts_inference.py"),
    Path("/content/Amphion/models/vocoders/vocoder_inference.py")
]

for file_path in search_files:

    print("\n" + "=" * 65)
    print("FILE:", file_path)
    print("=" * 65)

    if not file_path.exists():
        print("File not found. Searching for the actual file...")

        matches = list(
            Path("/content/Amphion").rglob(file_path.name)
        )

        for match in matches:
            print("FOUND:", match)

        continue

    lines = file_path.read_text(encoding="utf-8").splitlines()

    keywords = [
        "class TTSInference",
        "def load_checkpoint",
        "def load_model",
        "torch.load",
        "def synthesis",
        "generator_state_dict",
        "load_state_dict"
    ]

    shown = set()

    for i, line in enumerate(lines):
        if any(word in line for word in keywords):

            start = max(0, i - 5)
            end = min(len(lines), i + 16)

            for j in range(start, end):
                if j not in shown:
                    print(f"{j + 1}: {lines[j]}")
                    shown.add(j)

            print("-" * 40)

In [ ]:
from pathlib import Path

search_root = Path("/content/Amphion")

print("Searching for the TTS inference class...\n")

for file in search_root.rglob("*.py"):
    try:
        content = file.read_text(errors="ignore")

        if "class TTSInference" in content:
            print("FOUND:", file)

            lines = content.splitlines()

            for i, line in enumerate(lines):
                if "class TTSInference" in line:
                    start = max(0, i - 5)
                    end = min(len(lines), i + 100)

                    for j in range(start, end):
                        print(f"{j + 1}: {lines[j]}")

                    break

    except Exception:
        pass


In [ ]:
from pathlib import Path

files_to_check = [
    Path("/content/Amphion/models/tts/base/tts_inferece.py"),
    Path("/content/Amphion/models/tts/fastspeech2/fs2_inference.py")
]

for file in files_to_check:
    print("\n" + "=" * 65)
    print("FILE:", file)
    print("=" * 65)

    lines = file.read_text().splitlines()

    for i, line in enumerate(lines):
        if "def _load_model" in line or "def load_model" in line:
            start = max(0, i - 5)
            end = min(len(lines), i + 45)

            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")

In [ ]:
import torch

fs2_path = (
    "/content/fastspeech2_ljspeech/checkpoint/"
    "epoch-0125_step-0097902_loss-1.074404/"
    "pytorch_model.bin"
)

hifi_path = (
    "/content/hifigan_ljspeech/checkpoints/"
    "step-0560000_loss-51.2373.pt"
)

print("Checking FastSpeech 2 checkpoint...")

fs2 = torch.load(
    fs2_path,
    map_location="cpu",
    weights_only=False
)

print("FastSpeech 2 keys:", list(fs2.keys()))

del fs2

print("\nChecking HiFi-GAN checkpoint...")

hifi = torch.load(
    hifi_path,
    map_location="cpu",
    weights_only=False
)

print("HiFi-GAN keys:", list(hifi.keys()))

del hifi

print("\nCheckpoint inspection completed.")

In [ ]:
import shutil
from pathlib import Path

source_config = Path("/content/hifigan_ljspeech/args.json")

source_checkpoint = Path(
    "/content/hifigan_ljspeech/checkpoints/"
    "step-0560000_loss-51.2373.pt"
)

target_folder = Path("/content/hifigan_inference")
target_folder.mkdir(parents=True, exist_ok=True)

shutil.copy2(source_config, target_folder / "args.json")

shutil.copy2(
    source_checkpoint,
    target_folder / source_checkpoint.name
)

print("HiFi-GAN inference folder prepared!\n")

for file in target_folder.iterdir():
    print(f"{file.name} — {file.stat().st_size / (1024**2):.2f} MB")

In [ ]:
print("PREPROCESS CONFIGURATION KEYS:")
print(list(fs2_config.preprocess.keys()))

print("\nMODEL CONFIGURATION KEYS:")
print(list(fs2_config.model.keys()))

print("\nPHONE-RELATED SETTINGS:")
for key, value in fs2_config.preprocess.items():
    if "phone" in key.lower() or "encoder" in key.lower():
        print(key, ":", value)

In [ ]:
from utils.util import load_config

hifigan_config = load_config(
    "/content/hifigan_ljspeech/args.json",
    lowercase=True
)

print("HiFi-GAN configuration")
print("Model type:", hifigan_config.model_type)
print("Sample rate:", hifigan_config.preprocess.sample_rate)
print("Mel channels:", hifigan_config.preprocess.n_mel)
print("FFT size:", hifigan_config.preprocess.n_fft)
print("Hop size:", hifigan_config.preprocess.hop_size)
print("Window size:", hifigan_config.preprocess.win_size)

print("\nFastSpeech 2 configuration")
print("Sample rate:", fs2_config.preprocess.sample_rate)
print("Mel channels:", fs2_config.preprocess.n_mel)
print("FFT size:", fs2_config.preprocess.n_fft)
print("Hop size:", fs2_config.preprocess.hop_size)
print("Window size:", fs2_config.preprocess.win_size)

In [ ]:
import os

os.chdir("/content/Amphion")

inference_file = "/content/Amphion/bins/tts/inference.py"

with open(inference_file, "r") as f:
    lines = f.readlines()

print("FASTSPEECH 2 INFERENCE ARGUMENTS\n")

for i, line in enumerate(lines):
    if any(term in line for term in [
        "add_argument",
        "def main",
        "def build_inference",
        "config"
    ]):
        print(f"{i+1}: {line.rstrip()}")

In [ ]:
inference_file = "/content/Amphion/bins/tts/inference.py"

with open(inference_file, "r") as f:
    lines = f.readlines()

print("OFFICIAL INFERENCE SCRIPT")
print("=" * 60)

for i in range(40, 165):
    print(f"{i+1}: {lines[i].rstrip()}")

In [ ]:
import os

search_dirs = [
    "/content/Amphion/egs/tts/FastSpeech2",
    "/content/Amphion/config"
]

print("SEARCHING FOR INFERENCE CONFIGURATIONS\n")

for search_dir in search_dirs:
    if not os.path.exists(search_dir):
        continue

    for root, dirs, files in os.walk(search_dir):
        for file in files:
            if file.endswith((".yaml", ".yml", ".json")):
                path = os.path.join(root, file)

                if "infer" in file.lower() or "config" in file.lower():
                    print(path)

print("\nFASTSPEECH 2 MODEL SELECTION")
with open("/content/Amphion/bins/tts/inference.py") as f:
    lines = f.readlines()

for i in range(18, 40):
    print(f"{i+1}: {lines[i].rstrip()}")

In [ ]:
from utils.util import load_config

official_config_path = (
    "/content/Amphion/egs/tts/FastSpeech2/exp_config.json"
)

official_config = load_config(
    official_config_path,
    lowercase=True
)

print("OFFICIAL FASTSPEECH 2 CONFIGURATION")
print("=" * 45)

print("Model type:", official_config.model_type)
print("Sample rate:", official_config.preprocess.sample_rate)
print("Mel channels:", official_config.preprocess.n_mel)
print("FFT size:", official_config.preprocess.n_fft)
print("Hop size:", official_config.preprocess.hop_size)
print("Window size:", official_config.preprocess.win_size)

print("\nDOWNLOADED CHECKPOINT CONFIGURATION")
print("=" * 45)

print("Model type:", fs2_config.model_type)
print("Sample rate:", fs2_config.preprocess.sample_rate)
print("Mel channels:", fs2_config.preprocess.n_mel)
print("FFT size:", fs2_config.preprocess.n_fft)
print("Hop size:", fs2_config.preprocess.hop_size)
print("Window size:", fs2_config.preprocess.win_size)

In [ ]:
inference_file = (
    "/content/Amphion/models/tts/fastspeech2/fs2_inference.py"
)

with open(inference_file, "r") as f:
    lines = f.readlines()

print("FASTSPEECH 2 MODEL INITIALIZATION")
print("=" * 55)

for i, line in enumerate(lines):
    if "def _build_model" in line or "def load_model" in line:
        start = max(0, i - 3)
        end = min(len(lines), i + 30)

        for j in range(start, end):
            print(f"{j+1}: {lines[j].rstrip()}")

        print("-" * 55)

In [ ]:
import os

# FastSpeech 2 checkpoint
fs2_checkpoint_dir = (
    "/content/fastspeech2_ljspeech/checkpoint"
)

# HiFi-GAN inference folder
vocoder_dir = "/content/hifigan_inference"

print("FASTSPEECH 2 CHECKPOINT")
print("=" * 50)

if os.path.isdir(fs2_checkpoint_dir):
    for item in os.listdir(fs2_checkpoint_dir):
        print(item)
else:
    print("Checkpoint directory not found!")

print("\nHIFIGAN INFERENCE FOLDER")
print("=" * 50)

if os.path.isdir(vocoder_dir):
    for item in os.listdir(vocoder_dir):
        print(item)
else:
    print("HiFi-GAN inference folder not found!")

print("\nOFFICIAL CHECKPOINT LOADING CODE")
print("=" * 50)

base_file = (
    "/content/Amphion/models/tts/base/tts_inferece.py"
)

with open(base_file, "r") as f:
    lines = f.readlines()

for i in range(140, min(180, len(lines))):
    print(f"{i+1}: {lines[i].rstrip()}")

In [ ]:
base_file = (
    "/content/Amphion/models/tts/base/tts_inferece.py"
)

with open(base_file, "r") as f:
    lines = f.readlines()

print("INFERENCE INITIALIZATION")
print("=" * 55)

for i in range(20, 125):
    print(f"{i+1}: {lines[i].rstrip()}")

In [ ]:
print("CHECKING REQUIRED INFERENCE SETTINGS")
print("=" * 50)

print("Training configuration exists:",
      hasattr(official_config, "train"))

if hasattr(official_config, "train"):
    print("Random seed exists:",
          hasattr(official_config.train, "random_seed"))

    if hasattr(official_config.train, "random_seed"):
        print("Random seed:",
              official_config.train.random_seed)

print("\nOutput directory will be:")
print("/content/tts_output")

print("\nFastSpeech 2 checkpoint directory:")
print("/content/fastspeech2_ljspeech")

print("\nHiFi-GAN directory:")
print("/content/hifigan_inference")

In [ ]:
%pip install lhotse

In [ ]:
%pip install encodec

In [ ]:
%pip install Unidecode

In [ ]:
%pip install g2p-en

In [ ]:
%pip install phonemizer

In [ ]:
import os

paths = [
    "/content/fastspeech2_ljspeech/LJSpeech/phone_energys/statistics.json",
    "/content/fastspeech2_ljspeech/LJSpeech/phone_pitches/statistics.json",
    "/content/fastspeech2_ljspeech/args.json",
]

for path in paths:
    print(f"{'FOUND' if os.path.exists(path) else 'MISSING'}: {path}")

In [ ]:
import os

paths = [
    "/content/fastspeech2_ljspeech/LJSpeech/phone_energys/statistics.json",
    "/content/fastspeech2_ljspeech/LJSpeech/phone_pitches/statistics.json",
    "/content/fastspeech2_ljspeech/args.json",
]

for path in paths:
    print(("FOUND" if os.path.exists(path) else "MISSING") + ": " + path)

In [ ]:
print("Current processed_dir:", official_config.preprocess.processed_dir)
print("Current phone pitch stats:", official_config.preprocess.processed_dir + "/phone_pitches/statistics.json")
print("Current phone energy stats:", official_config.preprocess.processed_dir + "/phone_energys/statistics.json")

print("\nPretrained model directory:")
print("/content/fastspeech2_ljspeech/LJSpeech")

In [ ]:
official_config.preprocess.processed_dir = (
    "/content/fastspeech2_ljspeech/LJSpeech"
)

print("Updated processed_dir:", official_config.preprocess.processed_dir)

In [ ]:
official_config.preprocess.processed_dir = "/content/fastspeech2_ljspeech"

print("Corrected processed_dir:", official_config.preprocess.processed_dir)

In [ ]:
print("Initializing FastSpeech 2...")

fs2_inference = FastSpeech2Inference(
    args=args,
    cfg=official_config
)

print("FastSpeech 2 initialized successfully!")

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Model evaluation mode:", not fs2_inference.model.training)
print("Model device:", next(fs2_inference.model.parameters()).device)

In [ ]:
import os

vocoder_dir = "/content/hifigan_inference"

print("HiFi-GAN directory:", vocoder_dir)

for filename in os.listdir(vocoder_dir):
    filepath = os.path.join(vocoder_dir, filename)
    print(
        filename,
        f"({os.path.getsize(filepath) / (1024**2):.2f} MB)"
        if os.path.isfile(filepath) else "(directory)"
    )

In [ ]:
from utils.util import load_config
import glob
import os

vocoder_dir = "/content/hifigan_inference"

vocoder_config = load_config(
    os.path.join(vocoder_dir, "args.json"),
    lowercase=True
)

vocoder_checkpoints = glob.glob(
    os.path.join(vocoder_dir, "*.pt")
)

print("HiFi-GAN configuration loaded:", vocoder_config is not None)
print("Checkpoint found:", len(vocoder_checkpoints) > 0)
print("Checkpoint:", os.path.basename(vocoder_checkpoints[0]))
print("Sample rate:", vocoder_config.preprocess.sample_rate)
print("Mel channels:", vocoder_config.preprocess.n_mel)

In [ ]:
import torch
import os

checkpoint_path = vocoder_checkpoints[0]

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))
print("Checkpoint keys:", list(checkpoint.keys()))

if "generator_state_dict" in checkpoint:
    print("Generator weights: FOUND")
    print("Number of generator parameters:", len(checkpoint["generator_state_dict"]))
else:
    print("Generator weights: NOT FOUND")

In [ ]:
import inspect

print("FastSpeech 2 inference methods:\n")

for name, method in inspect.getmembers(
    FastSpeech2Inference,
    predicate=inspect.isfunction
):
    if not name.startswith("_"):
        print(name, inspect.signature(method))

In [ ]:
import inspect

print(
    inspect.getsource(
        FastSpeech2Inference.inference_for_single_utterance
    )
)

In [ ]:
args.text = "Hello! Welcome to my text to speech synthesis project."

args.pitch_control = 1.0
args.energy_control = 1.0
args.duration_control = 1.0

print("Test sentence:", args.text)
print("Pitch control:", args.pitch_control)
print("Energy control:", args.energy_control)
print("Duration control:", args.duration_control)
print("HiFi-GAN directory:", args.vocoder_dir)

In [ ]:
import nltk

nltk.download("averaged_perceptron_tagger_eng")
nltk.download("punkt_tab")

print("NLTK language resources downloaded.")

In [ ]:
import torch
import soundfile as sf
import os
from IPython.display import Audio, display

print("Generating speech...")

with torch.no_grad():
    generated_audio = fs2_inference.inference_for_single_utterance()

# Convert generated waveform to NumPy
audio_array = generated_audio.detach().cpu().numpy().squeeze()

# Save audio
output_dir = "/content/tts_output"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "first_generated_speech.wav")

sf.write(
    output_path,
    audio_array,
    22050
)

print("Speech generated successfully!")
print("Audio duration:", round(len(audio_array) / 22050, 2), "seconds")
print("Saved at:", output_path)

# Play generated audio
display(Audio(audio_array, rate=22050))

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

audio_path = "/content/tts_output/first_generated_speech.wav"

audio_data, sample_rate = sf.read(audio_path)

print("File verified:", audio_path)
print("Sample rate:", sample_rate)
print("Audio duration:", round(len(audio_data) / sample_rate, 2), "seconds")
print("Audio shape:", audio_data.shape)

display(Audio(audio_data, rate=sample_rate))

In [ ]:
import os
import shutil

project_drive = "/content/drive/MyDrive/Text_to_Speech_Project"

# Create permanent storage folders
backup_dir = os.path.join(project_drive, "saved_models")
os.makedirs(backup_dir, exist_ok=True)

# Save FastSpeech 2 model
shutil.copytree(
    "/content/fastspeech2_ljspeech",
    os.path.join(backup_dir, "fastspeech2_ljspeech"),
    dirs_exist_ok=True
)

# Save HiFi-GAN model
shutil.copytree(
    "/content/hifigan_ljspeech",
    os.path.join(backup_dir, "hifigan_ljspeech"),
    dirs_exist_ok=True
)

# Save generated speech
audio_backup = os.path.join(
    project_drive,
    "first_generated_speech.wav"
)

shutil.copy2(
    "/content/tts_output/first_generated_speech.wav",
    audio_backup
)

print("Backup completed!")
print("Saved models:", backup_dir)
print("Saved audio:", audio_backup)

In [ ]:
import os

print("FASTSPEECH 2 MODEL")
print("Model loaded:", "fs2_inference" in globals())

print("\nMODEL CHECKPOINTS")
print(
    "FastSpeech 2:",
    os.path.exists("/content/fastspeech2_ljspeech/checkpoint")
)
print(
    "HiFi-GAN:",
    os.path.exists("/content/hifigan_inference/step-0560000_loss-51.2373.pt")
)

In [ ]:
import importlib.util

packages = ["fastapi", "uvicorn"]

for package in packages:
    if importlib.util.find_spec(package):
        print(f"{package}: Already installed")
    else:
        print(f"{package}: Not installed")

In [ ]:
# new Create a separate Colab API application
from fastapi import FastAPI
from pydantic import BaseModel

colab_app = FastAPI(title="Colab Text-to-Speech API")

class SpeechRequest(BaseModel):
    text: str

@colab_app.get("/")
def home():
    return {"status": "Colab TTS server is ready"}

@colab_app.get("/health")
def health():
    return {
        "status": "healthy",
        "fastspeech2_loaded": "fs2_inference" in globals()
    }

print("Colab API application created successfully!")

In [ ]:
import inspect

print("INFERENCE FUNCTION SIGNATURE:")
print(inspect.signature(fs2_inference.inference_for_single_utterance))

print("\nINFERENCE FUNCTION SOURCE:")
print(inspect.getsource(fs2_inference.inference_for_single_utterance))

In [ ]:
import torch
import numpy as np

print("Audio type:", type(generated_audio))

if isinstance(generated_audio, torch.Tensor):
    print("Audio shape:", generated_audio.shape)
    print("Audio device:", generated_audio.device)
    print("Audio dtype:", generated_audio.dtype)
elif isinstance(generated_audio, np.ndarray):
    print("Audio shape:", generated_audio.shape)
    print("Audio dtype:", generated_audio.dtype)
else:
    print("Audio value:", generated_audio)

In [ ]:
import importlib.util

print("soundfile installed:", importlib.util.find_spec("soundfile") is not None)
print("Sample rate:", official_config.preprocess.sample_rate)

In [ ]:
import io
import threading
import soundfile as sf
from fastapi import HTTPException
from fastapi.responses import Response
from pydantic import BaseModel

# Prevent two requests from changing the model input simultaneously
inference_lock = threading.Lock()

class SynthesisRequest(BaseModel):
    text: str
    pitch_control: float = 1.0
    energy_control: float = 1.0
    duration_control: float = 1.0


@colab_app.post("/synthesize")
def synthesize_speech(request: SynthesisRequest):

    if not request.text.strip():
        raise HTTPException(
            status_code=400,
            detail="Text cannot be empty"
        )

    try:
        with inference_lock:

            # Set the input text and controls
            fs2_inference.args.text = request.text
            fs2_inference.args.pitch_control = request.pitch_control
            fs2_inference.args.energy_control = request.energy_control
            fs2_inference.args.duration_control = request.duration_control

            # Generate speech
            audio = fs2_inference.inference_for_single_utterance()

            # Convert tensor into NumPy audio
            audio_numpy = audio.detach().cpu().numpy()

            # Convert audio into a WAV file in memory
            audio_buffer = io.BytesIO()

            sf.write(
                audio_buffer,
                audio_numpy,
                22050,
                format="WAV",
                subtype="PCM_16"
            )

            audio_buffer.seek(0)

        return Response(
            content=audio_buffer.getvalue(),
            media_type="audio/wav",
            headers={
                "Content-Disposition": "attachment; filename=speech.wav"
            }
        )

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

print("Speech synthesis endpoint created successfully!")

In [ ]:
import threading
import uvicorn

def start_api_server():
    uvicorn.run(
        colab_app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

api_thread = threading.Thread(
    target=start_api_server,
    daemon=True
)

api_thread.start()

print("Colab API server starting...")

In [ ]:
import requests
import time

time.sleep(3)

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
import requests
from IPython.display import Audio, display

response = requests.post(
    "http://127.0.0.1:8000/synthesize",
    json={
        "text": "Hello! This is my text to speech mobile application."
    }
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("content-type"))

if response.status_code == 200:
    with open("/content/api_test.wav", "wb") as file:
        file.write(response.content)

    print("WAV file saved successfully!")
    display(Audio("/content/api_test.wav"))
else:
    print("Error:", response.text)

In [ ]:
import soundfile as sf
import os

audio_path = "/content/api_test.wav"

audio_data, sample_rate = sf.read(audio_path)

print("File exists:", os.path.exists(audio_path))
print("File size:", round(os.path.getsize(audio_path) / 1024, 2), "KB")
print("Sample rate:", sample_rate)
print("Audio shape:", audio_data.shape)
print("Duration:", round(len(audio_data) / sample_rate, 2), "seconds")
print("Audio valid:", len(audio_data) > 0)

In [ ]:
import shutil

cloudflared_path = shutil.which("cloudflared")

if cloudflared_path:
    print("Cloudflared is installed:", cloudflared_path)
else:
    print("Cloudflared is not installed.")

In [ ]:
import os
import subprocess

download_url = (
    "https://github.com/cloudflare/cloudflared/"
    "releases/latest/download/cloudflared-linux-amd64"
)

subprocess.run(
    ["wget", "-q", download_url, "-O", "/usr/local/bin/cloudflared"],
    check=True
)

os.chmod("/usr/local/bin/cloudflared", 0o755)

print("Cloudflared installation completed!")

subprocess.run(
    ["cloudflared", "--version"],
    check=True
)

In [ ]:
import subprocess
import time

tunnel_log = open("/content/cloudflared.log", "w")

tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:8000"
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

time.sleep(10)

with open("/content/cloudflared.log", "r") as file:
    logs = file.read()

print(logs)

In [ ]:
import requests

public_url = input("Enter the current Cloudflare tunnel URL: ").strip()

response = requests.get(
    f"{public_url}/health",
    timeout=30
)

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
tunnel_process.terminate()
tunnel_process.wait(timeout=10)

print("Public tunnel stopped.")
print("Colab API server is still running.")

In [ ]:
import secrets

API_KEY = secrets.token_urlsafe(32)

print("Your private API key:")
print("API key is configured; its value is intentionally hidden.")

In [ ]:
import secrets

API_KEY = secrets.token_urlsafe(32)

print("New private API key generated.")
print("Copy it directly into a secure local note.")
print("Do not paste it into this chat.")

In [ ]:
import secrets
import io
import threading
import soundfile as sf

from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel

secure_app = FastAPI(
    title="Secure Text-to-Speech API",
    version="1.0.0"
)

class SecureSynthesisRequest(BaseModel):
    text: str
    pitch_control: float = 1.0
    energy_control: float = 1.0
    duration_control: float = 1.0


@secure_app.get("/health")
def secure_health():
    return {
        "status": "healthy",
        "fastspeech2_loaded": True
    }


@secure_app.post("/synthesize")
def secure_synthesize(
    request: SecureSynthesisRequest,
    x_api_key: str = Header(...)
):
    if not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(
            status_code=401,
            detail="Invalid API key"
        )

    if not request.text.strip():
        raise HTTPException(
            status_code=400,
            detail="Text cannot be empty"
        )

    try:
        with inference_lock:
            fs2_inference.args.text = request.text
            fs2_inference.args.pitch_control = request.pitch_control
            fs2_inference.args.energy_control = request.energy_control
            fs2_inference.args.duration_control = request.duration_control

            audio = fs2_inference.inference_for_single_utterance()
            audio_numpy = audio.detach().cpu().numpy()

            audio_buffer = io.BytesIO()

            sf.write(
                audio_buffer,
                audio_numpy,
                22050,
                format="WAV",
                subtype="PCM_16"
            )

            audio_buffer.seek(0)

        return Response(
            content=audio_buffer.getvalue(),
            media_type="audio/wav",
            headers={
                "Content-Disposition": "attachment; filename=speech.wav"
            }
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print("Secure API created successfully!")

In [ ]:
import threading
import uvicorn

def start_secure_server():
    uvicorn.run(
        secure_app,
        host="0.0.0.0",
        port=8001,
        log_level="info"
    )

secure_thread = threading.Thread(
    target=start_secure_server,
    daemon=True
)

secure_thread.start()

print("Secure API server starting on port 8001...")

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8001/health",
    timeout=10
)

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
response = requests.post(
    "http://127.0.0.1:8001/synthesize",
    json={
        "text": "Testing API security."
    },
    timeout=10
)

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
import subprocess
import time

secure_tunnel_log = open(
    "/content/secure_cloudflared.log",
    "w"
)

secure_tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:8001"
    ],
    stdout=secure_tunnel_log,
    stderr=subprocess.STDOUT
)

time.sleep(10)

with open("/content/secure_cloudflared.log", "r") as file:
    print(file.read())

In [ ]:
import requests

secure_public_url = input("Enter the current secure Cloudflare tunnel URL: ").strip()

response = requests.get(
    f"{secure_public_url}/health",
    timeout=30
)

print("Status code:", response.status_code)
print("Response:", response.json())

In [ ]:
import requests

test_response = requests.post(
    "http://127.0.0.1:8001/synthesize",
    headers={"X-API-Key": API_KEY},
    json={"text": "Testing secure API authentication."},
    timeout=180
)

print("Status code:", test_response.status_code)
print("Content type:", test_response.headers.get("content-type"))

In [ ]:
import secrets

API_KEY = secrets.token_urlsafe(32)

print("NEW API KEY:")
print("API key is configured; its value is intentionally hidden.")

In [ ]:
import os

model_path = "/content/drive/MyDrive/Text_to_Speech_Project/saved_models"

print("Saved model folders:")
print(os.listdir(model_path))

In [ ]:
import os

print("Amphion folder exists:", os.path.exists("/content/Amphion"))

In [ ]:
!git clone https://github.com/open-mmlab/Amphion.git /content/Amphion

In [ ]:
import sys

print("Python version:", sys.version)

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available yet.")

In [ ]:
import os

for folder in [
    "fastspeech2_ljspeech",
    "hifigan_ljspeech"
]:
    path = f"/content/drive/MyDrive/Text_to_Speech_Project/saved_models/{folder}"

    print(f"\n{folder}:")
    for filename in os.listdir(path):
        print(" -", filename)

In [ ]:
import os

path = "/content/drive/MyDrive/Text_to_Speech_Project/saved_models/fastspeech2_ljspeech/checkpoint"

for root, dirs, files in os.walk(path):
    for filename in files:
        print(os.path.join(root, filename))

In [ ]:
import os

path = "/content/drive/MyDrive/Text_to_Speech_Project/saved_models/hifigan_ljspeech/checkpoints"

for root, dirs, files in os.walk(path):
    for filename in files:
        print(os.path.join(root, filename))

In [ ]:
import os

base = "/content/drive/MyDrive/Text_to_Speech_Project/saved_models"

for model in ["fastspeech2_ljspeech", "hifigan_ljspeech"]:
    path = os.path.join(base, model, "args.json")

    print(f"\n--- {model} ---")

    with open(path, "r") as f:
        print(f.read()[:1000])


In [ ]:
!pip install json5

In [ ]:
import os
import json5

base = "/content/drive/MyDrive/Text_to_Speech_Project/saved_models"

for model in ["fastspeech2_ljspeech", "hifigan_ljspeech"]:
    path = os.path.join(base, model, "args.json")

    with open(path, "r") as f:
        config = json5.load(f)

    print(f"\n{model}")
    print("Configuration loaded successfully!")
    print("Available settings:", list(config.keys()))


In [ ]:
!pip install ruamel.yaml

In [ ]:
import sys
sys.path.insert(0, "/content/Amphion")

from utils.util import load_config

print("Amphion configuration loader imported successfully!")

In [ ]:
import os

for path in [
    "/content/Amphion/config/fs2.json",
    "/content/Amphion/config/base.json"
]:
    print(path, "->", os.path.exists(path))

In [ ]:
import os

os.chdir("/content/Amphion")

print("Current working directory:", os.getcwd())

In [ ]:
fs2_config = load_config(fastspeech2_path)

print("FastSpeech 2 configuration loaded!")
print("Model type:", fs2_config["model_type"])
print("Dataset:", fs2_config["dataset"])

In [ ]:
hifigan_path = (
    "/content/drive/MyDrive/Text_to_Speech_Project/"
    "saved_models/hifigan_ljspeech/args.json"
)

hifigan_config = load_config(hifigan_path)

print("HiFi-GAN configuration loaded!")
print("Model type:", hifigan_config["model_type"])
print("Dataset:", hifigan_config["dataset"])

In [ ]:
import os

checkpoint_paths = [
    "/content/drive/MyDrive/Text_to_Speech_Project/saved_models/fastspeech2_ljspeech/checkpoint/epoch-0125_step-0097902_loss-1.074404/pytorch_model.bin",
    "/content/drive/MyDrive/Text_to_Speech_Project/saved_models/hifigan_ljspeech/checkpoints/step-0560000_loss-51.2373.pt"
]

for path in checkpoint_paths:
    print("\nFile:", os.path.basename(path))
    print("Exists:", os.path.isfile(path))

    if os.path.isfile(path):
        print("Size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

In [ ]:
import torch
import sys

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
import os

for root, dirs, files in os.walk("/content/Amphion"):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

    for file in files:
        if "fastspeech2" in file.lower():
            print(os.path.join(root, file))

In [ ]:
import os

for root, dirs, files in os.walk("/content/Amphion"):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

    for file in files:
        if "fastspeech2" in file.lower():
            print(os.path.join(root, file))

In [ ]:
!find /content/Amphion -iname '*fastspeech2*' -not -path '*/.git/*'

In [ ]:
import os

folder = "/content/Amphion/models/tts/fastspeech2"

for file in os.listdir(folder):
    print(file)

In [ ]:
!pip install Unidecode

In [ ]:
import inspect
from models.tts.fastspeech2.fs2 import FastSpeech2

print("FastSpeech 2 imported successfully!")
print("Constructor:", inspect.signature(FastSpeech2))


In [ ]:
!pip install lhotse

In [ ]:
!pip install encodec

In [ ]:
!pip install g2p-en

In [ ]:
import inspect
from models.tts.fastspeech2.fs2_inference import FastSpeech2Inference

print("Inference class imported successfully!")
print("Constructor:", inspect.signature(FastSpeech2Inference))

In [ ]:
import sys
import torch
import os

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not connected yet.")

print("\nGoogle Drive folder exists:",
      os.path.isdir("/content/drive/MyDrive"))

print("Amphion folder exists:",
      os.path.isdir("/content/Amphion"))

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

base = "/content/drive/MyDrive/Text_to_Speech_Project/saved_models"

print("Saved model folders:")

for folder in os.listdir(base):
    print(" -", folder)

In [ ]:
!git clone https://github.com/open-mmlab/Amphion.git /content/Amphion


In [ ]:
import os
import sys

os.chdir("/content/Amphion")

if "/content/Amphion" not in sys.path:
    sys.path.insert(0, "/content/Amphion")

print("Working directory:", os.getcwd())
print("Amphion Python path configured!")

In [ ]:
import importlib.util

packages = [
    "ruamel.yaml",
    "json5",
    "unidecode",
    "lhotse",
    "encodec",
    "g2p_en",
    "tgt",
    "librosa",
    "torchaudio",
    "accelerate",
    "transformers",
    "praat_parselmouth",
    "phonemizer",
    "soundfile",
]

print("Dependency check:\n")

for package in packages:
    try:
        available = importlib.util.find_spec(package) is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        available = False

    print(f"{package}: {'Installed' if available else 'Missing'}")

In [ ]:
!pip install ruamel.yaml json5 Unidecode lhotse encodec g2p-en tgt praat-parselmouth phonemizer==3.2.1

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available.")


In [ ]:
import inspect

from models.tts.fastspeech2.fs2_inference import FastSpeech2Inference

print("FastSpeech2Inference imported successfully!")
print("Constructor:", inspect.signature(FastSpeech2Inference))

In [ ]:
import inspect

print(inspect.getsource(FastSpeech2Inference.__init__))

In [ ]:
import inspect

for name, method in inspect.getmembers(
    FastSpeech2Inference,
    predicate=inspect.isfunction
):
    print(name, inspect.signature(method))

In [ ]:
import inspect

print(inspect.getsource(FastSpeech2Inference._load_model))

In [ ]:
import inspect
from models.tts.base.tts_inferece import TTSInference

print(inspect.getsource(TTSInference.__init__))

In [ ]:
import os

fastspeech2_dir = (
    "/content/drive/MyDrive/Text_to_Speech_Project/"
    "saved_models/fastspeech2_ljspeech"
)

checkpoint_path = (
    fastspeech2_dir +
    "/checkpoint/epoch-0125_step-0097902_loss-1.074404"
    "/pytorch_model.bin"
)

print("Model directory exists:", os.path.isdir(fastspeech2_dir))
print("Checkpoint exists:", os.path.isfile(checkpoint_path))
print("Checkpoint size (MB):", round(
    os.path.getsize(checkpoint_path) / (1024 * 1024), 2
) if os.path.isfile(checkpoint_path) else "Not found")

In [ ]:
import os

print("FastSpeech 2 config:",
      os.path.isfile(fastspeech2_dir + "/args.json"))

hifigan_dir = (
    "/content/drive/MyDrive/Text_to_Speech_Project/"
    "saved_models/hifigan_ljspeech"
)

print("HiFi-GAN directory exists:",
      os.path.isdir(hifigan_dir))

print("HiFi-GAN config:",
      os.path.isfile(hifigan_dir + "/args.json"))

In [ ]:
from utils.util import load_config

fs2_config = load_config(
    os.path.join(fastspeech2_dir, "args.json")
)

hifigan_config = load_config(
    os.path.join(hifigan_dir, "args.json")
)

print("FastSpeech 2 configuration loaded successfully!")
print("HiFi-GAN configuration loaded successfully!")

In [ ]:
print("FastSpeech 2 configuration:")
print(fs2_config)

In [ ]:
import json5
import json

with open(args_path, "r") as f:
    fs2_args_data = json5.load(f)

print("Saved FastSpeech 2 arguments:")
print(json.dumps(fs2_args_data, indent=2))

In [ ]:
import inspect
from models.tts.fastspeech2.fs2_inference import FastSpeech2Inference

print(inspect.getsource(FastSpeech2Inference.inference_for_single_utterance))

In [ ]:
import os

for root, dirs, files in os.walk(hifigan_dir):
    level = root.replace(hifigan_dir, "").count(os.sep)
    if level > 2:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(root) + "/")

    for file in files:
        print("  " * (level + 1) + file)

In [ ]:
print(hifigan_config)

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected")

In [ ]:
!git clone https://github.com/open-mmlab/Amphion.git /content/Amphion

In [ ]:
import os
import sys

os.chdir("/content/Amphion")

if "/content/Amphion" not in sys.path:
    sys.path.insert(0, "/content/Amphion")

print("Working directory:", os.getcwd())
print("Amphion Python path configured!")

In [ ]:
import importlib.util

packages = {
    "ruamel.yaml": "ruamel",
    "json5": "json5",
    "unidecode": "unidecode",
    "lhotse": "lhotse",
    "encodec": "encodec",
    "g2p_en": "g2p_en",
    "tgt": "tgt",
    "librosa": "librosa",
    "torchaudio": "torchaudio",
    "accelerate": "accelerate",
    "transformers": "transformers",
    "praat_parselmouth": "praat_parselmouth",
    "phonemizer": "phonemizer",
    "soundfile": "soundfile",
}

for package, module in packages.items():
    try:
        installed = importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        installed = False

    print(f"{package}: {'Installed' if installed else 'Missing'}")

In [ ]:
!pip install ruamel.yaml json5 Unidecode lhotse encodec g2p-en tgt praat-parselmouth phonemizer==3.2.1

In [ ]:
import torch
import importlib.util

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not detected")

packages = [
    "ruamel",
    "json5",
    "unidecode",
    "lhotse",
    "encodec",
    "g2p_en",
    "tgt",
    "praat_parselmouth",
    "phonemizer",
]

print("\nDependency check:")
for package in packages:
    print(package, ":", "Installed" if importlib.util.find_spec(package) else "Missing")

In [ ]:
!pip install praat-parselmouth

In [ ]:
import inspect
from models.tts.fastspeech2.fs2_inference import FastSpeech2Inference

print("FastSpeech 2 inference imported successfully!")
print("Constructor:", inspect.signature(FastSpeech2Inference))

In [ ]:
import inspect
from models.tts.fastspeech2 import fs2_inference

print("Available functions and classes:\n")

for name, obj in inspect.getmembers(fs2_inference):
    if inspect.isfunction(obj) or inspect.isclass(obj):
        print(name)

In [ ]:
import inspect
from models.tts.fastspeech2.fs2_inference import FastSpeech2Inference

print(inspect.getsource(FastSpeech2Inference.__init__))

In [ ]:
import inspect
from models.tts.fastspeech2.fs2_inference import FastSpeech2Inference

print(inspect.getsource(FastSpeech2Inference._load_model))

In [ ]:
import inspect
from models.tts.fastspeech2.fs2_inference import FastSpeech2Inference

print(inspect.getsource(FastSpeech2Inference._parse_vocoder))

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

project_dir = "/content/drive/MyDrive/Text_to_Speech_Project"

print("Project folder exists:", os.path.exists(project_dir))

if os.path.exists(project_dir):
    print("\nProject folder contents:")
    for item in os.listdir(project_dir):
        print("-", item)

In [ ]:
import os

saved_models_dir = os.path.join(project_dir, "saved_models")

print("Saved models folder exists:", os.path.exists(saved_models_dir))

for root, dirs, files in os.walk(saved_models_dir):
    level = root.replace(saved_models_dir, "").count(os.sep)

    if level > 2:
        dirs[:] = []
        continue

    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  {file}")

In [ ]:
import os

fs2_checkpoint_dir = os.path.join(
    project_dir,
    "saved_models",
    "fastspeech2_ljspeech",
    "checkpoint"
)

print("FastSpeech 2 checkpoint files:")

for root, dirs, files in os.walk(fs2_checkpoint_dir):
    for file in files:
        if file.endswith((".bin", ".safetensors")):
            path = os.path.join(root, file)
            print("File:", path)
            print("Size:", round(os.path.getsize(path) / (1024**2), 2), "MB")

In [ ]:
import os
import shutil

hifigan_dir = os.path.join(
    project_dir,
    "saved_models",
    "hifigan_ljspeech"
)

hifigan_inference_dir = "/content/hifigan_inference"

os.makedirs(hifigan_inference_dir, exist_ok=True)

shutil.copy2(
    os.path.join(hifigan_dir, "args.json"),
    hifigan_inference_dir
)

shutil.copy2(
    os.path.join(
        hifigan_dir,
        "checkpoints",
        "step-0560000_loss-51.2373.pt"
    ),
    hifigan_inference_dir
)

print("HiFi-GAN inference folder prepared:")
print(os.listdir(hifigan_inference_dir))

In [ ]:
fastspeech2_dir = os.path.join(
    project_dir,
    "saved_models",
    "fastspeech2_ljspeech"
)

hifigan_dir = hifigan_inference_dir

print("FastSpeech 2:", fastspeech2_dir)
print("FastSpeech 2 exists:", os.path.exists(fastspeech2_dir))

print("\nHiFi-GAN:", hifigan_dir)
print("HiFi-GAN exists:", os.path.exists(hifigan_dir))

In [ ]:
from utils.util import load_config

fs2_config = load_config(
    os.path.join(fastspeech2_dir, "args.json")
)

print("FastSpeech 2 configuration loaded!")
print("Model type:", fs2_config.model_type)
print("Sample rate:", fs2_config.preprocess.sample_rate)
print("Mel channels:", fs2_config.preprocess.n_mel)
print("Phone extractor:", fs2_config.preprocess.phone_extractor)

In [ ]:
hifigan_config = load_config(
    os.path.join(hifigan_dir, "args.json"),
    lowercase=True
)

print("HiFi-GAN configuration loaded!")
print("Model type:", hifigan_config.model_type)
print("Sample rate:", hifigan_config.preprocess.sample_rate)
print("Mel channels:", hifigan_config.preprocess.n_mel)

In [ ]:
import os

search_file = "/content/Amphion"

for root, dirs, files in os.walk(search_file):
    dirs[:] = [
        d for d in dirs
        if d not in [".git", "__pycache__"]
    ]

    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)

            if "inference" in file.lower() or file.lower() == "infer.py":
                print(path)


In [ ]:
inference_file = "/content/Amphion/bins/tts/inference.py"

with open(inference_file, "r") as file:
    source = file.read()

print(source)

In [ ]:
from types import SimpleNamespace

output_dir = "/content/tts_generated_audio"
os.makedirs(output_dir, exist_ok=True)

args = SimpleNamespace(
    mode="single",
    acoustics_dir=fastspeech2_dir,
    checkpoint_path=None,
    vocoder_dir=hifigan_dir,
    output_dir=output_dir,
    log_level="warning",
    text="Hello! Welcome to my Text to Speech mobile application.",
    pitch_control=1.0,
    energy_control=1.0,
    duration_control=1.0,
    speaker_name=None
)

print("Inference arguments prepared!")
print("Mode:", args.mode)
print("Text:", args.text)
print("Output folder:", args.output_dir)
print("FastSpeech 2 path exists:", os.path.exists(args.acoustics_dir))
print("HiFi-GAN path exists:", os.path.exists(args.vocoder_dir))

In [ ]:
import os

search_dirs = [
    project_dir,
    "/content/Amphion"
]

print("Searching for statistics.json...\n")

for search_dir in search_dirs:
    for root, dirs, files in os.walk(search_dir):
        if "statistics.json" in files:
            print(os.path.join(root, "statistics.json"))

In [ ]:
print(fs2_config.preprocess)

In [ ]:
fs2_config.preprocess.processed_dir = fastspeech2_dir

print("Updated processed directory:")
print(fs2_config.preprocess.processed_dir)

print("\nEnergy statistics exists:",
      os.path.isfile(os.path.join(
          fastspeech2_dir,
          fs2_config.preprocess.phone_energy_dir,
          "statistics.json"
      )))

print("Pitch statistics exists:",
      os.path.isfile(os.path.join(
          fastspeech2_dir,
          fs2_config.preprocess.phone_pitch_dir,
          "statistics.json"
      )))

In [ ]:
fs2_config.preprocess.processed_dir = os.path.join(
    fastspeech2_dir,
    "LJSpeech"
)

print("Processed directory:", fs2_config.preprocess.processed_dir)

print(
    "Energy statistics exists:",
    os.path.isfile(os.path.join(
        fs2_config.preprocess.processed_dir,
        fs2_config.preprocess.phone_energy_dir,
        "statistics.json"
    ))
)

print(
    "Pitch statistics exists:",
    os.path.isfile(os.path.join(
        fs2_config.preprocess.processed_dir,
        fs2_config.preprocess.phone_pitch_dir,
        "statistics.json"
    ))
)

In [ ]:
fs2_config.preprocess.processed_dir = fastspeech2_dir

energy_path = os.path.join(
    fastspeech2_dir,
    "LJSpeech",
    "phone_energys",
    "statistics.json"
)

pitch_path = os.path.join(
    fastspeech2_dir,
    "LJSpeech",
    "phone_pitches",
    "statistics.json"
)

print("Processed directory:", fs2_config.preprocess.processed_dir)
print("Energy statistics:", os.path.exists(energy_path))
print("Pitch statistics:", os.path.exists(pitch_path))

In [ ]:
print("Initializing FastSpeech 2...")

fs2_inference = FastSpeech2Inference(
    args,
    fs2_config
)

print("\nFastSpeech 2 initialized successfully!")
print("Device:", fs2_inference.device)

In [ ]:
import nltk

nltk.download("averaged_perceptron_tagger_eng")

print("NLTK English tagger installation completed!")

In [ ]:
print("Generating speech...")

generated_audio = fs2_inference.inference_for_single_utterance()

print("\nSpeech generation completed!")
print("Audio shape:", generated_audio.shape)

In [ ]:
import soundfile as sf
import numpy as np

output_audio_path = os.path.join(
    project_dir,
    "second_generated_speech.wav"
)

audio_array = generated_audio.detach().cpu().numpy()

sf.write(
    output_audio_path,
    audio_array,
    22050
)

print("Audio saved successfully!")
print("File:", output_audio_path)
print("Duration:", round(len(audio_array) / 22050, 2), "seconds")
print("Sample rate: 22050 Hz")

In [ ]:
from IPython.display import Audio, display

display(Audio(filename=output_audio_path))

In [ ]:
import os

folders = {
    "FastSpeech 2": os.path.join(
        project_dir, "saved_models", "fastspeech2_ljspeech"
    ),
    "HiFi-GAN": os.path.join(
        project_dir, "saved_models", "hifigan_ljspeech"
    )
}

for model, folder in folders.items():
    print(f"\n{model}:")
    print("Folder exists:", os.path.exists(folder))
    print("Configuration exists:", os.path.exists(
        os.path.join(folder, "args.json")
    ))

In [ ]:
print("Secure API application available:", "secure_app" in globals())
print("API key available:", "API_KEY" in globals())

In [ ]:
!pip install fastapi uvicorn pycloudflared

In [ ]:
import secrets

API_KEY = secrets.token_urlsafe(32)

print("New secure API key generated successfully!")
print("Key length:", len(API_KEY))
print("Key hidden for security.")

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel, Field
import threading

secure_app = FastAPI(
    title="Secure Text-to-Speech API",
    description="FastSpeech 2 + HiFi-GAN speech synthesis",
    version="1.0.0"
)

inference_lock = threading.Lock()


class SpeechRequest(BaseModel):
    text: str = Field(min_length=1)
    pitch_control: float = 1.0
    energy_control: float = 1.0
    duration_control: float = 1.0


@secure_app.get("/health")
def health():
    return {"status": "healthy", "model": "FastSpeech 2 + HiFi-GAN"}


@secure_app.post("/synthesize")
def synthesize(
    request: SpeechRequest,
    x_api_key: str = Header(None)
):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")

    args.text = request.text
    args.pitch_control = request.pitch_control
    args.energy_control = request.energy_control
    args.duration_control = request.duration_control

    try:
        with inference_lock:
            audio = fs2_inference.inference_for_single_utterance()

        import numpy as np
        import soundfile as sf
        import io

        audio_array = audio.detach().cpu().numpy()
        buffer = io.BytesIO()
        sf.write(buffer, audio_array, 22050, format="WAV")

        return Response(
            content=buffer.getvalue(),
            media_type="audio/wav"
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print("Secure FastAPI application created successfully!")
print("Health endpoint: /health")
print("Speech endpoint: /synthesize")
print("API key authentication: Enabled")

In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

config = uvicorn.Config(
    secure_app,
    host="0.0.0.0",
    port=8001,
    log_level="warning"
)

api_server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=api_server.run,
    daemon=True
)

api_thread.start()

print("Secure API server started on port 8001!")

In [ ]:
!pip install pycloudflared

In [ ]:
from pycloudflared import try_cloudflare

tunnel = try_cloudflare(port=8001)

print("Cloudflare tunnel started!")
print(tunnel)

In [ ]:
import secrets

API_KEY = secrets.token_urlsafe(32)

print("New API key generated successfully!")

In [ ]:
print("API key ready:", bool(API_KEY))
print("FastSpeech 2 ready:", "fs2_inference" in globals())
print("Model arguments ready:", "args" in globals())

In [ ]:
import os

project_dir = "/content/drive/MyDrive/Text_to_Speech_Project"

fastspeech2_dir = os.path.join(
    project_dir, "saved_models", "fastspeech2_ljspeech"
)

hifigan_dir = os.path.join(
    project_dir, "saved_models", "hifigan_ljspeech"
)

print("FastSpeech 2 folder exists:", os.path.exists(fastspeech2_dir))
print("HiFi-GAN folder exists:", os.path.exists(hifigan_dir))
print("FastSpeech 2 config exists:",
      os.path.exists(os.path.join(fastspeech2_dir, "args.json")))
print("HiFi-GAN config exists:",
      os.path.exists(os.path.join(hifigan_dir, "args.json")))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

project_dir = "/content/drive/MyDrive/Text_to_Speech_Project"

print("Project folder exists:", os.path.exists(project_dir))

if os.path.exists(project_dir):
    print("\nProject folder contents:")
    for item in os.listdir(project_dir):
        print("-", item)

In [ ]:
import os

saved_models_dir = os.path.join(project_dir, "saved_models")

for model in ["fastspeech2_ljspeech", "hifigan_ljspeech"]:
    model_path = os.path.join(saved_models_dir, model)

    print(f"\n{model}:")
    print("Folder exists:", os.path.exists(model_path))

    if os.path.exists(model_path):
        for root, dirs, files in os.walk(model_path):
            for file in files:
                if file.endswith((".bin", ".pt", ".json")):
                    print(os.path.join(root, file))

In [ ]:
!git clone https://github.com/open-mmlab/Amphion.git /content/Amphion

In [ ]:
!pip install ruamel.yaml json5 Unidecode lhotse encodec g2p-en tgt praat-parselmouth phonemizer==3.2.1


In [ ]:
import os
import sys

os.chdir("/content/Amphion")

if "/content/Amphion" not in sys.path:
    sys.path.insert(0, "/content/Amphion")

print("Working directory:", os.getcwd())
print("Amphion Python path configured successfully!")

In [ ]:
from models.tts.fastspeech2.fs2_inference import FastSpeech2Inference

print("FastSpeech 2 inference module imported successfully!")

In [ ]:
import os

project_dir = "/content/drive/MyDrive/Text_to_Speech_Project"

fastspeech2_dir = os.path.join(
    project_dir, "saved_models", "fastspeech2_ljspeech"
)

hifigan_saved_dir = os.path.join(
    project_dir, "saved_models", "hifigan_ljspeech"
)

print("FastSpeech 2 folder exists:", os.path.isdir(fastspeech2_dir))
print("FastSpeech 2 config exists:", os.path.isfile(os.path.join(fastspeech2_dir, "args.json")))

print("HiFi-GAN folder exists:", os.path.isdir(hifigan_saved_dir))
print("HiFi-GAN config exists:", os.path.isfile(os.path.join(hifigan_saved_dir, "args.json")))

In [ ]:
import os

fs2_checkpoint_dir = os.path.join(
    fastspeech2_dir,
    "checkpoint",
    "epoch-0125_step-0097902_loss-1.074404"
)

hifigan_checkpoint_dir = os.path.join(
    hifigan_saved_dir,
    "checkpoints"
)

print("FastSpeech 2 checkpoint:",
      os.path.isfile(os.path.join(fs2_checkpoint_dir, "pytorch_model.bin")))

print("HiFi-GAN checkpoint:",
      os.path.isfile(os.path.join(hifigan_checkpoint_dir, "step-0560000_loss-51.2373.pt")))

In [ ]:
print("FastSpeech 2 configuration:")
print(fs2_config)

print("\nHiFi-GAN configuration:")
print(hifigan_config)

In [ ]:
print("FastSpeech 2 model:", fs2_config.model_type)
print("FastSpeech 2 sample rate:", fs2_config.preprocess.sample_rate)
print("FastSpeech 2 mel channels:", fs2_config.preprocess.n_mel)

print("\nHiFi-GAN model:", hifigan_config.model_type)
print("HiFi-GAN sample rate:", hifigan_config.preprocess.sample_rate)
print("HiFi-GAN mel channels:", hifigan_config.preprocess.n_mel)

print(
    "\nAudio configurations compatible:",
    fs2_config.preprocess.sample_rate == hifigan_config.preprocess.sample_rate
    and fs2_config.preprocess.n_mel == hifigan_config.preprocess.n_mel
)

In [ ]:
import os
import shutil

hifigan_dir = "/content/hifigan_inference"

os.makedirs(hifigan_dir, exist_ok=True)

shutil.copy2(
    os.path.join(hifigan_saved_dir, "args.json"),
    os.path.join(hifigan_dir, "args.json")
)

shutil.copy2(
    os.path.join(
        hifigan_saved_dir,
        "checkpoints",
        "step-0560000_loss-51.2373.pt"
    ),
    os.path.join(
        hifigan_dir,
        "step-0560000_loss-51.2373.pt"
    )
)

print("HiFi-GAN inference folder prepared successfully!")
print("Files:", os.listdir(hifigan_dir))

In [ ]:
print("FastSpeech 2 checkpoint directory:")
print(fs2_checkpoint_dir)

print("\nFastSpeech 2 checkpoint files:")
print(os.listdir(fs2_checkpoint_dir))

print("\nHiFi-GAN inference files:")
print(os.listdir(hifigan_dir))

print("\nFastSpeech 2 configuration loaded:", fs2_config is not None)
print("HiFi-GAN configuration loaded:", hifigan_config is not None)

In [ ]:
import inspect

print(inspect.signature(FastSpeech2Inference))

In [ ]:
import inspect

print(inspect.getsource(FastSpeech2Inference.__init__))

In [ ]:
import inspect

parent_class = FastSpeech2Inference.__mro__[1]

print("Parent class:", parent_class)
print("Parent class module:", parent_class.__module__)
print("\nParent class source file:")
print(inspect.getfile(parent_class))

In [ ]:
import inspect

print(inspect.getsource(parent_class.__init__))


In [ ]:
import inspect

print(inspect.getsource(FastSpeech2Inference.inference_for_single_utterance))

In [ ]:
import inspect

print(inspect.getsource(FastSpeech2Inference.inference_for_single_utterance))

In [ ]:
print("Multi-speaker training:", fs2_config.train.multi_speaker_training)
print("Use speaker ID:", fs2_config.preprocess.use_spkid)
print("Random seed:", fs2_config.train.random_seed)
print("Phone extractor:", fs2_config.preprocess.phone_extractor)

In [ ]:
from argparse import Namespace
import os

args = Namespace(
    mode="single",
    acoustics_dir=fastspeech2_dir,
    checkpoint_path=None,
    vocoder_dir=hifigan_dir,
    output_dir=os.path.join(project_dir, "generated_outputs"),
    log_level="INFO",
    text="Hello! Welcome to my Text to Speech application.",
    pitch_control=1.0,
    energy_control=1.0,
    duration_control=1.0
)

os.makedirs(args.output_dir, exist_ok=True)

print("Inference arguments prepared successfully!")
print("Mode:", args.mode)
print("FastSpeech 2 directory:", args.acoustics_dir)
print("HiFi-GAN directory:", args.vocoder_dir)
print("Output directory:", args.output_dir)
print("Text:", args.text)

In [ ]:
print("Configured processed directory:")
print(fs2_config.preprocess.processed_dir)

print("\nActual statistics directory:")
print(os.path.join(fastspeech2_dir, "LJSpeech"))

print("\nAvailable statistics folders:")
print(os.listdir(os.path.join(fastspeech2_dir, "LJSpeech")))

In [ ]:
fs2_config.preprocess.processed_dir = fastspeech2_dir

print("Updated processed directory:")
print(fs2_config.preprocess.processed_dir)

print("\nEnergy statistics exist:",
      os.path.exists(os.path.join(
          fs2_config.preprocess.processed_dir,
          "LJSpeech",
          "phone_energys",
          "statistics.json"
      )))

print("Pitch statistics exist:",
      os.path.exists(os.path.join(
          fs2_config.preprocess.processed_dir,
          "LJSpeech",
          "phone_pitches",
          "statistics.json"
      )))

In [ ]:
fs2_inference = FastSpeech2Inference(args, fs2_config)

print("\nFastSpeech 2 initialized successfully!")
print("Device:", fs2_inference.device)

In [ ]:
import nltk

nltk.download("averaged_perceptron_tagger_eng")

print("NLTK English tagger download completed!")

In [ ]:
generated_audio = fs2_inference.inference_for_single_utterance()

print("Speech generation completed!")
print("Generated audio shape:", generated_audio.shape)
print("Audio device:", generated_audio.device)

In [ ]:
import soundfile as sf
import os

output_audio_path = os.path.join(
    project_dir,
    "restored_model_test.wav"
)

audio_array = generated_audio.detach().cpu().numpy()

sf.write(output_audio_path, audio_array, 22050)

print("Audio saved successfully!")
print("File:", output_audio_path)
print("Duration:", round(len(audio_array) / 22050, 2), "seconds")
print("Sample rate: 22050 Hz")

In [ ]:
from IPython.display import Audio, display

display(Audio(audio_array, rate=22050))

In [ ]:
import importlib.util

packages = ["fastapi", "uvicorn", "pycloudflared", "httpx"]

for package in packages:
    status = "Installed" if importlib.util.find_spec(package) else "Missing"
    print(f"{package}: {status}")

print("\nAPI key available:", "API_KEY" in globals())
print("FastSpeech 2 available:", "fs2_inference" in globals())

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel, Field
import threading
import numpy as np
import soundfile as sf
import io

secure_app = FastAPI(
    title="Secure Text-to-Speech API",
    description="FastSpeech 2 + HiFi-GAN speech synthesis",
    version="1.0.0"
)

inference_lock = threading.Lock()

class SpeechRequest(BaseModel):
    text: str = Field(min_length=1)
    pitch_control: float = 1.0
    energy_control: float = 1.0
    duration_control: float = 1.0

@secure_app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": "FastSpeech 2 + HiFi-GAN"
    }

@secure_app.post("/synthesize")
def synthesize(
    request: SpeechRequest,
    x_api_key: str = Header(None)
):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")

    try:
        with inference_lock:
            args.text = request.text
            args.pitch_control = request.pitch_control
            args.energy_control = request.energy_control
            args.duration_control = request.duration_control

            audio = fs2_inference.inference_for_single_utterance()

        audio_array = audio.detach().cpu().numpy()

        buffer = io.BytesIO()
        sf.write(buffer, audio_array, 22050, format="WAV")

        return Response(
            content=buffer.getvalue(),
            media_type="audio/wav"
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("Secure FastAPI application created successfully!")

In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

config = uvicorn.Config(
    secure_app,
    host="0.0.0.0",
    port=8001,
    log_level="warning"
)

api_server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=api_server.run,
    daemon=True
)

api_thread.start()

print("FastAPI server started on port 8001!")

In [ ]:
from pycloudflared import try_cloudflare

tunnel = try_cloudflare(port=8001)

print("Cloudflare tunnel created successfully!")
print(tunnel)

In [ ]:
print("API key is configured; its value is intentionally hidden.")

In [ ]:
import os
import shutil

project_dir = "/content/drive/MyDrive/Text_to_Speech_Project"

source = os.path.join(project_dir, "saved_models")
archive = "/content/tts_models_backup"

shutil.make_archive(
    archive,
    "zip",
    source
)

print("Model backup ZIP created successfully!")
print("File:", archive + ".zip")
print("Size:", round(os.path.getsize(archive + ".zip") / (1024**2), 2), "MB")

In [ ]:
from google.colab import files

files.download("/content/tts_models_backup.zip")